In [22]:
pip install requests beautifulsoup4 lxml

In [23]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

In [28]:


def get_soup(url):
    """
    Récupère le contenu HTML d'une page Wikipédia et le parse avec BeautifulSoup.

    :param url: URL complète de la page Wikipédia
    :return: objet BeautifulSoup
    """
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; WikiScraper/1.0; +https://example.com)"
    }
    response = requests.get(url, headers=headers, timeout=15)
    response.raise_for_status()
    # 'lxml' est plus rapide et robuste que le parser html.parser par défaut
    soup = BeautifulSoup(response.content, "lxml")
    return soup

def get_title(soup):
    """
    Extrait le titre principal de l'article Wikipédia.
    """
    # La balise <h1> avec l'id "firstHeading" contient le titre de l'article
    h1 = soup.find("h1", id="firstHeading")
    return h1.get_text(strip=True) if h1 else None


def get_sections_text(soup):
    """
    Extrait le texte de chaque section de l'article, en associant chaque
    en-tête (h2, h3) à la liste des paragraphes qui le suivent.

    :return: dictionnaire {titre_section: [paragraphes]}
    """
    content_div = soup.find("div", id="mw-content-text")
    if not content_div:
        return {}

    # On récupère uniquement les en-têtes h2 et h3 (les sections principales)
    headings = content_div.find_all(["h2", "h3"])

    sections = {}
    for heading in headings:
        # Nettoyer le titre (supprimer les [modifier] éventuels)
        title = heading.get_text(strip=True).replace("[modifier]", "").strip()
        if not title or title.lower() in ("notes et références", "voir aussi"):
            # On peut ignorer certaines sections si souhaité
            pass

        paragraphs = []
        # On parcourt les frères suivants jusqu'au prochain en-tête
        for sibling in heading.find_next_siblings():
            if sibling.name in ("h2", "h3"):
                break
            if sibling.name == "p":
                text = sibling.get_text(strip=True)
                if text:
                    paragraphs.append(text)

        if paragraphs:
            sections[title] = paragraphs

    return sections

def get_internal_links(soup, base_url="https://en.wikipedia.org/wiki/Naruto"):
    """
    Collecte tous les liens internes vers d'autres pages Wikipédia
    (exclut les liens vers : catégories, fichiers, pages spéciales, ancres, etc.)

    :return: liste de liens absolus uniques
    """
    links = set()

    content_div = soup.find("div", id="mw-content-text")
    if not content_div:
        return []

    for a in content_div.find_all("a", href=True):
        href = a["href"]

        # On ne garde que les liens internes /wiki/...
        if not href.startswith("/wiki/"):
            continue

        # Exclure les pages spéciales / méta
        if href.startswith(("/wiki/Catégorie:", "/wiki/Fichier:",
                            "/wiki/Aide:", "/wiki/Spécial:",
                            "/wiki/Wikipédia:", "/wiki/Discussion:",
                            "/wiki/Portail:", "/wiki/Modèle:")):
            continue

        full_url = urljoin(base_url, href)

        # Retirer les ancres (#section) pour éviter les doublons
        full_url = full_url.split("#")[0]

        links.add(full_url)

    return sorted(links)

def scrape_wikipedia(url):
    """
    Fonction consolidée : à partir d'une URL Wikipédia, retourne un dictionnaire
    contenant le titre, les sections (en-tête → paragraphes) et les liens internes.

    :param url: URL de la page Wikipédia
    :return: dict {"url", "title", "sections", "internal_links"}
    """
    soup = get_soup(url)

    return {
        "url": url,
        "title": get_title(soup),
        "sections": get_sections_text(soup),
        "internal_links": get_internal_links(soup),
    }



In [29]:
if __name__ == "__main__":
      url_test = "https://en.wikipedia.org/wiki/Naruto"
      result = scrape_wikipedia(url_test)

      print("=" * 60)
      print("URL      :", result["url"])
      print("TITRE    :", result["title"])
      print("=" * 60)

      print("\n--- SECTIONS ET PARAGRAPHES ---\n")
      for section, paragraphs in result["sections"].items():
          print(f"## {section}")
          for p in paragraphs[:2]:   # On affiche les 2 premiers paragraphes par section
              print(f"   - {p[:150]}...")
          print()

      print("=" * 60)
      print(f"LIENS INTERNES TROUVÉS : {len(result['internal_links'])}")
      print("=" * 60)
      for link in result["internal_links"][:10]:
          print(" ", link)

URL      : https://en.wikipedia.org/wiki/Naruto
TITRE    : Naruto

--- SECTIONS ET PARAGRAPHES ---

LIENS INTERNES TROUVÉS : 0
